# 🚀 Обучение Модели C+ (Multi-Scale CNN + Dual-Graph) — Google Colab

Этот блокнот предназначен для обучения самой продвинутой архитектуры: **Model C+**.

**SOTA улучшения:**
1. **Multi-scale Inception-style stem** (k=3,5,7,11) для разных частот ЭКГ
2. **Lead Positional Encoding** для сохранения идентичности отведений
3. **Dual-Graph Adaptive Attention**: Клинический граф + Адаптивное Cosine сходство по фичам!
4. **Transformer-style GNN Blocks** (Pre-LN, FFN)
5. **Multi-Readout Pooling** (Mean + Attention Gating)

Результаты синхронизируются напрямую с Google Диском.

In [1]:
# ШАГ 1: Подключаем Google Диск
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ШАГ 2: Переходим в папку проекта
import os

project_path = '/content/drive/MyDrive/ecg-diploma'  
if os.path.exists(project_path):
    os.chdir(project_path)
    print("✅ Успешно перешли в директорию:", os.getcwd())
else:
    print("❌ Папка не найдена.")
    
!ls -la

✅ Успешно перешли в директорию: /content/drive/MyDrive/ecg-diploma
total 58
-rw------- 1 root root  262 Dec 20 12:43 CODEOWNERS
-rw------- 1 root root 2284 Dec 20 12:43 CONTRIBUTING.md
drwx------ 2 root root 4096 Jan 12 17:22 data_preprocessed
drwx------ 2 root root 4096 Mar 23 13:48 experiments
drwx------ 2 root root 4096 Mar 26 09:43 .git
drwx------ 2 root root 4096 Dec 20 12:43 .github
-rw------- 1 root root 5064 Dec 20 12:43 .gitignore
drwx------ 2 root root 4096 Mar 26 10:25 notebooks
-rw------- 1 root root 5452 Jan 12 18:25 README.md
-rw------- 1 root root 1056 Jan 12 17:34 requirements.txt
drwx------ 3 root root 4096 Mar 28 11:49 results
-rw------- 1 root root   38 Dec 20 12:43 results_summary.csv
-rw------- 1 root root 6019 Jan 12 18:25 SETUP.md
drwx------ 2 root root 4096 Dec 20 12:43 splits
drwx------ 5 root root 4096 Mar 28 09:57 src
drwx------ 2 root root 4096 Jan 12 17:00 .venv


In [3]:
# ШАГ 3: Устанавливаем зависимости
!pip install -q -r requirements.txt
print("✅ Зависимости проверены/установлены!")

✅ Зависимости проверены/установлены!


In [ ]:
# ШАГ 5: ПОЛНОЦЕННОЕ ОБУЧЕНИЕ (Model C+)
# Улучшенные параметры для SOTA-результата (AdamW, Weight Decay, Warmup)
!python -m src.train --model model_c_plus \
    --epochs 80 \
    --patience 16 \
    --batch_size 64 \
    --accum_steps 2 \
    --lr 7e-4 \
    --weight_decay 2e-2 \
    --warmup_epochs 8 \
    --label_smoothing 0.05 \
    --num_workers 2 \
    --use_weighted_sampler \
    --output_dir /content/drive/MyDrive/ecg-diploma/results

✅ Global seed set: 42

  PTB-XL SOTA Training v2.0 — MODEL_C_PLUS
  Device          : cuda  [AMP ON]
  Batch size      : 64 × 2 accum = 128 effective
  LR schedule     : Warmup(8ep) → CosineAnnealing
  Label smoothing : 0.05
  Epochs          : 80  |  Patience: 16
  Output          : /content/drive/MyDrive/ecg-diploma/results/model_c_plus
  Resume          : NO ← новое обучение

─────────────────────────────────────────────────────────────────
✅ Global seed set: 42
✅ Loaded train split: 17418 samples
   Signal shape: (12, 1000)
   Classes: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
   Augmentation: ON
   Label distribution:
     NORM: 7596 (43.6%)
     MI: 4379 (25.1%)
     STTC: 4087 (23.5%)
     CD: 3907 (22.4%)
     HYP: 2119 (12.2%)
✅ Loaded val split: 2183 samples
   Signal shape: (12, 1000)
   Classes: ['NORM', 'MI', 'STTC', 'CD', 'HYP']
   Augmentation: OFF
   Label distribution:
     NORM: 955 (43.7%)
     MI: 540 (24.7%)
     STTC: 515 (23.6%)
     CD: 495 (22.7%)
     HYP: 268 (12.3

In [ ]:
# ШАГ 6: ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from pathlib import Path

possible_roots = [
    Path(os.getcwd()) / 'results' / 'model_c_plus',
    Path('/content/drive/MyDrive/ecg-diploma/results/model_c_plus')
]

log_path = None
test_path = None
for p in possible_roots:
    if (p / 'train_log.csv').exists():
        log_path = p / 'train_log.csv'
        test_path = p / 'test_results.json'
        print(f"✅ Логи найдены по пути: {p}")
        break

if log_path and log_path.exists():
    df_log = pd.read_csv(log_path)
    
    sns.set_theme(style="whitegrid")
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # --- График 1: Loss ---
    ax1.plot(df_log['epoch'], df_log['train_loss'], label='Train Loss', marker='o')
    ax1.plot(df_log['epoch'], df_log['val_loss'], label='Val Loss', marker='o')
    ax1.set_title('Динамика функции потерь (Loss)')
    ax1.set_xlabel('Эпоха')
    ax1.legend()

    # --- График 2: Метрики ---
    if 'val_macro_auc' in df_log.columns:
        ax2.plot(df_log['epoch'], df_log['val_macro_auc'], label='Val Macro AUROC', color='green', marker='s')
        best_auroc = df_log['val_macro_auc'].max()
        best_ep = df_log.loc[df_log['val_macro_auc'].idxmax(), 'epoch']
        ax2.axvline(best_ep, color='blue', linestyle='--', label=f'Best AUROC={best_auroc:.4f} (Ep {int(best_ep)})')
        ax2.set_title('AUROC')
        ax2.legend()

    plt.show()

    if test_path and test_path.exists():
        with open(test_path, 'r') as f:
            test_res = json.load(f)
        print("\n🏆 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ (TEST) 🏆")
        print(f" 🔸 Macro AUROC: {test_res.get('macro_auc', 0):.4f}")
        print(f" 🔸 Micro AUROC: {test_res.get('micro_auc', 0):.4f}")